# Overview

[link](https://docs.langchain.com/oss/python/langchain/multi-agent/index)

## Why multi-agent

- Context management:
    - Many tasks with long context each one.
    - Need to selectively surface needed context at each time.
- Distributted development:
    - Different teams work on different tasks / agents, with clear boundaries.
- Parallel execution:
    - Different tasks / agents run in parallel.
- Enforce sequential constraints: unlock capabilities only when conditions are met.

## Multi-agent vs single agent with multiple tools:
- Too many tools -> agent makes poor selection of tool to use.
- It may require too much context. 
    - Multi-agent especially interesting if context can be split into domains.


## Patterns

- Agents:
    - Each subagent is a tool for the supervisor agent.
    - All routing through centralized supervisor agent
        - Supervisor communicates separately with different subagents and with user.
        - Sub-agents don't communicate with user directly. 
        - They are stateless by default. Supervisor keeps state.
    - Supervisor decides dynamically when and how to call each subagent, or to not call any.
- Handoffs:
    - Depending on centralized state, one agent may transfer the task (or part of it?) to another one.
    - Tool (which may be agent?) update state variable that triggers
        - Routing
        - Configuration changes
        - Switching agents or adjusting current agent's tools and prompt.
- Skills:
    - Agent loads skills as required. 
        - Each skill may be loaded by dedicated agent.
    - Each skill may be a specialized prompt or knowledge loaded on-demand.
- Router: 
    - User's query is classified into one of X tasks. Relevant agent is called.
    - Output from different agents is synthesized.
- Custom:
    - Combination of the above. 
    - Each agent may be node in final graph.

## Choosing a pattern

| Pattern | Distributed development | Parallel | Multi-hop (1) | Direct user interaction (2) |
| --- | --- | --- | --- | --- | 
| Subagent | 5 | 5 | 5 | 1 |
| Handsoff | 0 | 0 | 5 | 5 |
| Skills | 0 | 3 | 5 | 5 |
| Router | 3 | 5 | 0 | 3 |

- (1) Multi-hop: can multiple subagents be called in a series?
- (2) Direct interaction: can subagents converse directly with user?

## Subagents diagram

```mermaid
flowchart LR
User["User Request"] --> Supervisor
Supervisor --> Subagent1
Subagent1 --> Supervisor
Supervisor --> Subagent2
Subagent2 --> Supervisor
Supervisor --> Subagent3
Subagent3 --> Supervisor
Supervisor --> Response
``` 

## Subagents one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Supervisor
    participant Coffee subagent
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Supervisor: I want coffee
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Supervisor->>Coffee subagent: coffee_subagent()
    note right of Supervisor: Call 2
    end

    Coffee subagent->>buy_coffee tool: buy_coffee()

    rect rgb(0, 0, 0)
    buy_coffee tool-->>Coffee subagent: Done
    note right of Coffee subagent: Call 3
    end

    rect rgb(0, 0, 0)
    Coffee subagent-->>Supervisor: bought coffee
    note right of Supervisor: Call 4
    end

    Supervisor-->>User: I bought your cofee

```

## Handoffs diagram

```mermaid
flowchart LR
User["User Request"] --> A["Agent A"]
A["Agent A"] <--> B["Agent B"]
A <--> C["Agent C"]
A --> Response
B <--> C
B --> Response
C --> Response
``` 

## Handoffs one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Main Agent
    participant Coffee Agent
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Main Agent: I want coffee
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Main Agent->>Coffee Agent: transfer_to_coffee_agent()
    note right of Main Agent: Call 2
    end

    Coffee Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0, 0, 0)
    buy_coffee tool-->>Coffee Agent: Done
    note right of Coffee Agent: Call 3
    end

    Coffee Agent-->>User: I bought your cofee

```

## Skills diagram

```mermaid
flowchart LR
User["User Request"] --> M["Agent"]
M --> A["Skill A"]
M --> B["Skill B"]
M --> C["Skill C"]
M --> Response
``` 

## Skills one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant load_skill tool
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Agent: I want coffee
    note right of User: Call 1
    end

    Agent->>load_skill tool: load_skill("coffee")

    rect rgb(0,0,0)
    load_skill tool-->>Agent: Coffee skill context
    note right of Agent: Call 2
    end

    Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0,0,0)
    buy_coffee tool-->>Agent: Done
    note right of Agent: Call 3
    end

    Agent-->>User: I bought your cofee

```

## Router diagram

```mermaid
flowchart LR
User["User Request"] --> Router
Router --> A["Agent A"]
Router --> B["Agent B"]
Router --> C["Agent C"]
A --> Synthesis
B --> Synthesis
C --> Synthesis
Synthesis --> Response
``` 

## Router one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Router
    participant Coffee Agent
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Router: I want coffee
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Router->>Coffee Agent: coffee_agent()
    note right of Router: Call 2
    end

    Coffee Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0, 0, 0)
    buy_coffee tool-->>Coffee Agent: Done
    note right of Coffee Agent: Call 3
    end

    Coffee Agent-->>User: I bought your cofee

```

## Repeat request

- Subagents: statless: 4 * 2 = 8 calls
- Handoffs: coffee agent keeps state, no handoff, calls tool directly (call 1), after done (call 2) responds to user : 3+2 = 5
- Skills: coffee skills not required. 
    - After receiving repeat request (call 1), agent calls buy tool.
    - After response "Done", (call 2) responds to user.
    - 3+2=5
- Router: 
    - After receiving repeat request (call 1), router calls Coffee Agent (call 2).
    - After response "Done", (call 3) Coffee Agent responds to user.
    - 3+3=6

## Subagents multi-language comparison

```mermaid
sequenceDiagram
    rect rgb(0, 0, 0)
    User->>Supervisor: Compare Python, JS, Ruby
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Supervisor->>Python Expert: get_analysis()
    note right of Supervisor: Call 2
    end

    Python Expert-->>Supervisor: python analysis

    rect rgb(0, 0, 0)
    Supervisor->>JS Expert: get_analysis()
    note right of Supervisor: Call 4
    end

    JS Expert-->>Supervisor: js analysis

    rect rgb(0, 0, 0)
    Supervisor->>Ruby Expert: get_analysis()
    note right of Supervisor: Call 5
    end

    Ruby Expert-->>Supervisor: ruby analysis

    rect rgb(0, 0, 0)
    Supervisor-->>User: Comparison
    note right of Supervisor: Call 6
    end

```

## Handoffs one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Main Agent
    participant Coffee Agent
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Main Agent: I want coffee
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Main Agent->>Coffee Agent: transfer_to_coffee_agent()
    note right of Main Agent: Call 2
    end

    Coffee Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0, 0, 0)
    buy_coffee tool-->>Coffee Agent: Done
    note right of Coffee Agent: Call 3
    end

    Coffee Agent-->>User: I bought your cofee

```

## Skills one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant load_skill tool
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Agent: I want coffee
    note right of User: Call 1
    end

    Agent->>load_skill tool: load_skill("coffee")

    rect rgb(0,0,0)
    load_skill tool-->>Agent: Coffee skill context
    note right of Agent: Call 2
    end

    Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0,0,0)
    buy_coffee tool-->>Agent: Done
    note right of Agent: Call 3
    end

    Agent-->>User: I bought your cofee

```

## Router one-shot request

```mermaid
sequenceDiagram
    participant User
    participant Router
    participant Coffee Agent
    participant buy_coffee tool

    rect rgb(0, 0, 0)
    User->>Router: I want coffee
    note right of User: Call 1
    end

    rect rgb(0, 0, 0)
    Router->>Coffee Agent: coffee_agent()
    note right of Router: Call 2
    end

    Coffee Agent->>buy_coffee tool: buy_coffee()

    rect rgb(0, 0, 0)
    buy_coffee tool-->>Coffee Agent: Done
    note right of Coffee Agent: Call 3
    end

    Coffee Agent-->>User: I bought your cofee

```

# Subagents

[link](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)

## Key characteristics

- As tools
- Stateless
- Centralized
- No direct user interaction btw subagent and user (possible through `interrupts`)
- Parallelizable

## When to use

- Multiple distinct domains
- Distributed development
- Parallelization
- No need of direct interaction btw subagent and user

For simpler cases with just few tools, use single-agent

## Design decisions

### Synchronous vs Asynchronous

- Synchronous: main agent needs *all* subagent results before proceeding with conversation with user.
- Asynchronous: the agent can keep conversation and have other requests or update the user while waiting for subagent results.

### Tool patterns

- Tool per agent: more setup but more customization.
- Single dispatch tool: 
    - Convention over configuration.
    - Good for: 
        - Distributed teams
        - Many agents.

## Context engineering

### Context spec

- Descriptive name of tool, arguments, intent, when to use it.
- For single dispatch tool, list subagents names:
    - In prompt
    - As Enumeration of tool name, if few subagents
    - As get_list tool if dynamic.

### Subagent inputs

- In subagent tool, we can get access to main agent state and:
    - Include messages from history (either all or latest)
    - Include any other state's specific field we consider relevant

### Subagent outputs

Ways to ensure the main agent gets necessary information backs from subagent:
- Prompt subagent: request explicitly to return required results in its last message.
- Add structure: 
    - format of output's message
    - fill additional state field via `Command`




# Handoffs

[link](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs)

## Overview

Key concept: 
- tools update state variable
    - Example: `active_agent` or `current_step`
- based on updated state, system (can be single agent) modify:
    - system prompt
    - list of tools
    - routing (next agent / step in graph)


## Key characteristics

- Direct interaction
- Stateful
- Tool-based transitions: tools update state to signal that we need to change btw states

## When to use

- Flow:
    - Enforce sequential constraints.
    - Multi-stage conversational flows.
        - Typical use case: custom support, collecting info from customer in specific order (warranty ID before processing refund).
- Direct interaction.
- No need of paralellization or distributed development.

## Implementation approaches

### Single agent + middleware

- Tools change state variable
- Middleware examine the state before agent being called
    - Modify its prompt or the list of tools

### Multiple agents

- Separate agents as separate nodes in graph.
- Tools make transitions by doing:
    ```python
    return Command (
        goto="next_agent", # to indicate next step
        graph=Command.PARENT, # to make the transition happen directly
        update=my_new_state, # optional
    )
    ```

## Context engineering

### Handling context between handoffs

Tool response needs to include:
- Last AIMessage that triggered the handoff
- Artificial ToolMessage that indicates new state and acknowledges handoff
- `graph=...`, `goto=...` (see above)

Typically in a `transfer_to_x_agent` tool function

## Returning control to the user

- When agent's turn finishes and we return control to the user, make sure the last message is an `AIMessage`

## Implementation considerations

- Context filtering:
    - part of message history, condensed / summarized or not...
- Token efficiency (related)
- Tool semantics:
    - Only transition tool
    - Doing more actions apart from transitioning (e.g., `transfer_to_sales()` also creates support ticket)

# Skills

[link](https://docs.langchain.com/oss/python/langchain/multi-agent/skills)

## Key characteristics

- Prompt-driven: skill information injected via prompt.
- On-demand: different skills are loaded at different times on demand.
- Distributted teams: easy to develop independently.
- Lightweight: simpler than full subagents.

## When to use

- Specialized knowledge for each skill, distinct domains.
- The main agent just needs to get this information. 

## Extending the pattern

- Dynamically tool loading:
    - As a skill is loaded, new relevant tools can be added. 
        - Example, database admin skill can also load register database, ... tools
- Hierarchical skills:
    - General skill can also load more specialized skills if needed
        - Example: data-science skill, then pandas skill, numpy skill, etc.

# Router

[link](https://docs.langchain.com/oss/python/langchain/multi-agent/router)

## Key characteristics

- Router decomposes the query.
- Zero or more specialized agents invoked in parallel.
- Results synthesized at the end.

## When to use
- Tasks well defined into verticals
- Need to summarize / synthesize result at the end.
- Need to query multiple sources in parallel.

## Single agent vs multiple agents

### Single agent

The query results only in one action, out of N possible actions, being executed by either another node in the graph, which can wrap an agent. It is like having N branches in the graph and just being able to go to one of them, not several of them in parallel. 

We use an edge in the graph which:

- classifies the query into the appropriate branch out of the N possible.
- outputs a `Command` result with `goto` indicating destination node.

## Multiple agents

The query can contain multiple separate topics, each being tackled by different agent or node in the graph.

We use a sort of edge function very similar to single agent except for the fact that this one returns `M` `Send` objects, one per topic in the query, instead of returning a single `Command` object.

```python
topics = get_query_topics (query)
return [Send(topic["agent"], topic["description"]) for topic in topics]
```

## Stateless vs stateful

- Typical implementation: stateless router. For single turn conversations.
- Multi-turn conversations: use stateful router using one of two mechanims.

### 1. Tool wrapper

The stateless router is wrapped as a tool and invoked by a stateful agent which maintains the conversation and state.

### 2. Full persistence

- The state is persisted in store. 
- The router takes the history and passes just the bits relevant for the destination agent, to manage context appropriately.